# Experiment 1: MusicBrainz API Exploration

**Goal:** Understand the data structure — what fields we get for artists, releases, and relationships, and how they map to our ontology.

**Key questions:**
- What does an artist record look like?
- How are genres/tags represented?
- How are relationships (member of, influenced by) structured?
- Where are the links to Wikipedia/Discogs/Wikidata?
- What do releases and tracks look like?

## Setup

In [1]:
import musicbrainzngs
import json

# Required by MusicBrainz — set a descriptive user agent
musicbrainzngs.set_useragent("KE-CW2-MusicHistory", "0.1", "lucas@example.com")

# Helper to pretty-print JSON responses
def pp(data):
    print(json.dumps(data, indent=2, default=str))

## 1. Search for an Artist

Let's start simple — search for an artist by name and see what comes back.

In [2]:
result = musicbrainzngs.search_artists(artist="David Bowie", limit=3)
pp(result)

{
  "artist-list": [
    {
      "id": "5441c29d-3602-4898-b1a1-b77fa23b8e50",
      "type": "Person",
      "ext:score": "100",
      "name": "David Bowie",
      "sort-name": "Bowie, David",
      "gender": "male",
      "area": {
        "id": "9d5dd675-3cf4-4296-9e39-67865ebee758",
        "type": "Subdivision",
        "name": "England",
        "sort-name": "England",
        "life-span": {
          "ended": "false"
        }
      },
      "begin-area": {
        "id": "61941444-5a83-43e2-ba48-4db7438e0f26",
        "type": "District",
        "name": "Brixton",
        "sort-name": "Brixton",
        "life-span": {
          "ended": "false"
        }
      },
      "end-area": {
        "id": "74e50e58-5deb-4b99-93a2-decbb365c07f",
        "type": "City",
        "name": "New York",
        "sort-name": "New York",
        "life-span": {
          "ended": "false"
        }
      },
      "disambiguation": "English singer\u2010songwriter",
      "ipi-list": [
        "0000396

**Things to note:**
- Each artist has an `id` (MBID) — a UUID we'll use as a URI in our KG
- `type` distinguishes `Person` vs `Group`
- `country` and `area` give geographic info
- `life-span` has `begin`/`end` dates
- `ext:score` shows search relevance

Let's grab the top result's MBID for further exploration.

In [5]:
artist_mbid = result["artist-list"][0]["id"]
artist_name = result["artist-list"][0]["name"]
print(f"Using: {artist_name} ({artist_mbid})")

Using: David Bowie (5441c29d-3602-4898-b1a1-b77fa23b8e50)


## 2. Full Artist Details + Relationships

Now let's fetch the full record with `includes` to get tags (genres), URL links, and artist-to-artist relationships.

In [6]:
artist_detail = musicbrainzngs.get_artist_by_id(
    artist_mbid,
    includes=["tags", "url-rels", "artist-rels"]
)["artist"]

pp(artist_detail)

{
  "id": "5441c29d-3602-4898-b1a1-b77fa23b8e50",
  "type": "Person",
  "name": "David Bowie",
  "sort-name": "Bowie, David",
  "gender": "Male",
  "country": "GB",
  "area": {
    "id": "9d5dd675-3cf4-4296-9e39-67865ebee758",
    "name": "England",
    "sort-name": "England",
    "iso-3166-2-code-list": [
      "GB-ENG"
    ]
  },
  "begin-area": {
    "id": "61941444-5a83-43e2-ba48-4db7438e0f26",
    "name": "Brixton",
    "sort-name": "Brixton"
  },
  "end-area": {
    "id": "74e50e58-5deb-4b99-93a2-decbb365c07f",
    "name": "New York",
    "sort-name": "New York"
  },
  "disambiguation": "English singer\u2010songwriter",
  "ipi": "00003960406",
  "ipi-list": [
    "00003960406",
    "00015471209"
  ],
  "isni-list": [
    "0000000114448576",
    "0000000458257298"
  ],
  "life-span": {
    "begin": "1947-01-08",
    "end": "2016-01-10",
    "ended": "true"
  },
  "artist-relation-list": [
    {
      "type": "collaboration",
      "type-id": "75c09861-6857-4ec0-9729-84eefde7fc86",

That's a lot of data. Let's break it down into the parts that matter for our ontology.

### 2a. Basic Info

Maps to: `Artist` entity with properties `name`, `type`, `country`, `birthPlace`, `activeYears`

In [8]:
print(f"Name:       {artist_detail['name']}")
print(f"Type:       {artist_detail.get('type', 'N/A')}")
print(f"Country:    {artist_detail.get('country', 'N/A')}")
print(f"Gender:     {artist_detail.get('gender', 'N/A')}")

life_span = artist_detail.get("life-span", {})
print(f"Born:       {life_span.get('begin', 'N/A')}")
print(f"Died:       {life_span.get('end', 'N/A')}")
print(f"Area:       {artist_detail.get('area', {}).get('name', 'N/A')}")
print(f"Birth Area: {artist_detail.get('begin-area', {}).get('name', 'N/A')}")

Name:       David Bowie
Type:       Person
Country:    GB
Gender:     Male
Born:       1947-01-08
Died:       2016-01-10
Area:       England
Birth Area: Brixton


### 2b. Tags / Genres

Maps to: `hasGenre` property. Tags are community-contributed and have a `count` (votes).

In [9]:
tags = artist_detail.get("tag-list", [])
print(f"Tags ({len(tags)}):\n")
for tag in sorted(tags, key=lambda t: int(t.get("count", 0)), reverse=True)[:15]:
    print(f"  {tag['name']:30s} (count: {tag.get('count', '?')})")

Tags (20):

  art rock                       (count: 22)
  glam rock                      (count: 22)
  alternative rock               (count: 12)
  pop                            (count: 10)
  art pop                        (count: 7)
  rock                           (count: 7)
  british                        (count: 6)
  uk                             (count: 6)
  new wave                       (count: 5)
  pop rock                       (count: 5)
  actors                         (count: 3)
  classic rock                   (count: 3)
  arrangers                      (count: 1)
  composers                      (count: 1)
  glam rock music                (count: 1)


### 2c. URL Relationships

These link to external resources — **this is how we cross-reference to Wikipedia, Discogs, and Wikidata.**

In [10]:
url_rels = artist_detail.get("url-relation-list", [])
print(f"URL relationships ({len(url_rels)}):\n")
for rel in url_rels:
    print(f"  {rel['type']:25s} → {rel['target']}")

URL relationships (81):

  allmusic                  → https://www.allmusic.com/artist/mn0000531986
  bandsintown               → https://www.bandsintown.com/a/951
  BBC Music page            → https://www.bbc.co.uk/music/artists/5441c29d-3602-4898-b1a1-b77fa23b8e50
  discography page          → https://www.illustrated-db-discography.nl/
  discography page          → https://www.sonymusic.co.jp/artist/DavidBowie/
  discography page          → http://www.davidbowiecatalogue.com/
  discogs                   → https://www.discogs.com/artist/10263
  fanpage                   → https://bowiesongs.wordpress.com/
  fanpage                   → https://www.bowiewonderworld.com/
  fanpage                   → https://www.davidbowieitalia.it/
  fanpage                   → http://www.bowiedownunder.com/
  fanpage                   → http://www.algonet.se/~bassman/
  fanpage                   → http://www.helden.org.uk/Number/1970list.htm
  fanpage                   → http://www.littleoogie.com/
  f

### 2d. Artist-to-Artist Relationships

Maps to: `influencedBy`, `memberOf`, `collaboratedWith`, etc.

In [11]:
artist_rels = artist_detail.get("artist-relation-list", [])
print(f"Artist relationships ({len(artist_rels)}):\n")
for rel in artist_rels[:20]:
    direction = rel.get("direction", "")
    target = rel.get("artist", {}).get("name", "Unknown")
    rel_type = rel.get("type", "unknown")
    attrs = ", ".join(rel.get("attribute-list", []))
    begin = rel.get("begin", "")
    end = rel.get("end", "")
    time = f" ({begin}–{end})" if begin or end else ""
    attr_str = f" [{attrs}]" if attrs else ""
    print(f"  [{direction:7s}] {rel_type:25s} → {target}{time}{attr_str}")

Artist relationships (66):

  [forward] collaboration             → Band Aid [minor]
  [backward] instrumental supporting musician → Carlos Alomar [guitar]
  [backward] involved with             → Cyrinda Foxe
  [backward] involved with             → Claudia Lennear
  [backward] involved with             → Jean Millington
  [backward] involved with             → Susan Sarandon
  [backward] involved with             → Amanda Lear
  [forward] married                   → Angela Bowie (1970-03-19–1980-02-08)
  [forward] married                   → Iman Mohamed Abdulmajid (1992-04-24–2016-01-10)
  [forward] member of band            → David Bowie & The Buzz (1966-02-10–1966-12-02) [lead vocals, saxophone]
  [forward] member of band            → The Arnold Corns [original]
  [forward] member of band            → The Manish Boys
  [forward] member of band            → Bewlay Bros.
  [forward] member of band            → Davy Jones & The Lower Third [original]
  [forward] member of band       

## 3. Releases (Albums)

Maps to: `Album` entity with `releaseDate`, `countryOfOrigin`

In [12]:
releases = musicbrainzngs.browse_releases(
    artist=artist_mbid,
    release_type=["album"],
    limit=10
)

print(f"Total albums: {releases.get('release-count', '?')}")
print(f"Showing first {len(releases['release-list'])}:\n")

for rel in releases["release-list"]:
    print(f"  {rel['title']}")
    print(f"    MBID:    {rel['id']}")
    print(f"    Date:    {rel.get('date', 'N/A')}")
    print(f"    Country: {rel.get('country', 'N/A')}")
    print(f"    Status:  {rel.get('status', 'N/A')}")
    print()

Total albums: 1456
Showing first 10:

  David Bowie
    MBID:    f4678eec-fabf-42a3-bcc5-99d2e243bc11
    Date:    1967-06
    Country: GB
    Status:  Official

  David Bowie
    MBID:    9d6d8272-55fd-4b7b-a71e-4d9df6ee8f00
    Date:    1967-06
    Country: GB
    Status:  Official

  David Bowie
    MBID:    63fb3f61-ae89-4a19-8c7d-1f288af9bd5e
    Date:    1967-08
    Country: US
    Status:  Official

  David Bowie
    MBID:    bfa03202-690b-411a-bf6a-baa78d74cfb7
    Date:    1967
    Country: US
    Status:  Official

  David Bowie
    MBID:    767ee61d-6395-4ba9-bf79-ab0bf6701669
    Date:    1969-11-04
    Country: GB
    Status:  Official

  Man of Words/Man of Music
    MBID:    63027a5d-8b94-4a6b-a3bb-cd5d55134214
    Date:    1969
    Country: US
    Status:  Official

  The Man Who Sold the World
    MBID:    8784de2f-5754-3bcf-8b2a-35d327ef74a1
    Date:    1970-11-04
    Country: US
    Status:  Official

  The World of David Bowie
    MBID:    7ae8fe60-6bae-4749-b09f-9

## 4. Tracks on an Album

Maps to: `Track` entity with `hasTrack` (Album → Track)

In [13]:
# Pick the first release from above
release_mbid = releases["release-list"][0]["id"]

release_detail = musicbrainzngs.get_release_by_id(
    release_mbid,
    includes=["recordings", "artist-credits"]
)["release"]

print(f"Album: {release_detail['title']}")
print(f"Date:  {release_detail.get('date', 'N/A')}\n")

for medium in release_detail.get("medium-list", []):
    print(f"Medium {medium.get('position', '?')} ({medium.get('format', 'unknown')}):\n")
    for track in medium.get("track-list", []):
        rec = track.get("recording", {})
        length_ms = rec.get("length")
        if length_ms:
            mins = int(length_ms) // 60000
            secs = (int(length_ms) % 60000) // 1000
            length_str = f"{mins}:{secs:02d}"
        else:
            length_str = "N/A"
        print(f"  {track.get('position', '?'):>2s}. {rec.get('title', 'Unknown'):40s} ({length_str})")

Album: David Bowie
Date:  1967-06

Medium 1 (12" Vinyl):

   1. Uncle Arthur                             (2:09)
   2. Sell Me a Coat                           (3:00)
   3. Rubber Band                              (2:19)
   4. Love You Till Tuesday                    (3:13)
   5. There Is a Happy Land                    (3:16)
   6. We Are Hungry Men                        (3:01)
   7. When I Live My Dream                     (3:25)
   8. Little Bombardier                        (3:26)
   9. Silly Boy Blue                           (3:54)
  10. Come and Buy My Toys                     (2:10)
  11. Join the Gang                            (2:20)
  12. She’s Got Medals                         (2:27)
  13. Maid of Bond Street                      (1:46)
  14. Please Mr. Gravedigger                   (2:38)


## 5. Band Membership — The Beatles

Maps to: `Band`, `memberOf`, `playsInstrument`

In [14]:
beatles = musicbrainzngs.search_artists(artist="The Beatles", limit=1)
beatles_mbid = beatles["artist-list"][0]["id"]

beatles_detail = musicbrainzngs.get_artist_by_id(
    beatles_mbid,
    includes=["artist-rels", "tags"]
)["artist"]

print(f"Name:   {beatles_detail['name']}")
print(f"Type:   {beatles_detail.get('type', 'N/A')}")
ls = beatles_detail.get('life-span', {})
print(f"Active: {ls.get('begin', '?')} – {ls.get('end', '?')}")

Name:   The Beatles
Type:   Group
Active: 1960 – 1970-04-10


In [16]:
# Filter for member-of-band relationships
members = [
    r for r in beatles_detail.get("artist-relation-list", [])
    if r.get("type") == "member of band"
]

print(f"Members ({len(members)}):\n")
for m in members:
    name = m.get("artist", {}).get("name", "Unknown")
    attrs = ", ".join(m.get("attribute-list", []))
    begin = m.get("begin", "?")
    end = m.get("end", "?")
    print(f"  {name:20s} ({begin} – {end})" + (f"  [{attrs}]" if attrs else ""))

Members (9):

  Stuart Sutcliffe     (1960-01 – 1961)  [bass guitar]
  Tommy Moore          (1960-05 – 1960-06)  [drums (drum set)]
  Norman Chapman       (1960-06 – 1960-06)  [drums (drum set)]
  Pete Best            (1960-08-12 – 1962-08-16)  [drums (drum set)]
  Chas Newby           (1960-12 – 1961-01)  [bass guitar]
  John Lennon          (1960 – 1969-09)  [guitar, lead vocals, original]
  George Harrison      (1960 – 1970-04-10)  [guitar, lead vocals, original]
  Paul McCartney       (1960 – 1970-04-10)  [bass guitar, lead vocals, original]
  Ringo Starr          (1962-08 – 1970-04-10)  [drums (drum set)]


In [17]:
# Beatles genres/tags
tags = beatles_detail.get("tag-list", [])
print("Tags/Genres:\n")
for tag in sorted(tags, key=lambda t: int(t.get("count", 0)), reverse=True)[:10]:
    print(f"  {tag['name']:30s} (count: {tag.get('count', '?')})")

Tags/Genres:

  rock                           (count: 45)
  pop                            (count: 28)
  pop rock                       (count: 23)
  british                        (count: 22)
  psychedelic pop                (count: 15)
  merseybeat                     (count: 14)
  british invasion               (count: 9)
  psychedelic rock               (count: 7)
  60s                            (count: 6)
  classic rock                   (count: 3)


## 6. Summary — Mapping to Our Ontology

Run this after exploring the data above. Record your observations.

| MusicBrainz Field | Our Ontology Entity/Property | Notes |
|---|---|---|
| `id` (MBID) | URI for any entity | `http://musicbrainz.org/artist/{mbid}` |
| `name` | `rdfs:label` or `foaf:name` | |
| `type` (Person/Group) | `Artist` vs `Band` class | |
| `country` / `area` | `countryOfOrigin` | |
| `begin-area` | `birthPlace` | |
| `life-span.begin/end` | `activeYears` or birth/death dates | |
| `gender` | `gender` property | |
| `tag-list[].name` | `hasGenre` → `Genre` | Use count as confidence |
| `artist-relation-list` (type=member of band) | `memberOf` | Has begin/end dates |
| `artist-relation-list` (type=influenced by) | `influencedBy` | |
| `url-relation-list` (type=wikipedia) | Cross-reference to Wikipedia | For text extraction |
| `url-relation-list` (type=discogs) | Cross-reference to Discogs | For sub-genre data |
| `url-relation-list` (type=wikidata) | Cross-reference to Wikidata | For linked data |
| `release-list[].title` | `Album` entity | |
| `release-list[].date` | `releaseDate` | |
| `recording.title` | `Track` entity | |
| `recording.length` | `duration` property | In milliseconds |

## Next Steps

- **Experiment 2**: Discogs API — compare fields, explore sub-genres and artist profiles
- **Experiment 3**: Wikipedia API — fetch article text for NER/LLM extraction
- **Experiment 4**: Cross-reference — follow the URL rels from MusicBrainz to Discogs and Wikipedia